# LandScan + FIRMS integrity check

Reason: the LandScan/FIRMS downloads were suspected to contain duplicate files saved under different year-stamped names (the source URL is sometimes year-agnostic). Stage B pulls LandScan as the population covariate and FIRMS as the active-fire covariate, so a silent duplication would have the population layer of one year stand in for three years (and FIRMS' archive+NRT bundle silently double-count records at the boundary date).

Run this notebook **before** wiring either dataset into the Stage B feature builder.

Checks performed:
1. **LandScan ZIPs** — byte-size + MD5 hash per zip, then internal payload comparison.
2. **LandScan rasters** — open the GeoTIFF inside each ZIP, compare bounds, value histogram, and a fingerprint hash. If any of these match across years, the file is the same year masquerading under another name.
3. **FIRMS archive + NRT** — date-range, geographic-bounds, and (lat, lon, ACQ_DATE) collisions between the two shapefiles. We expect the archive to end roughly where the NRT begins; a wide overlap means we will double-count fires.

In [1]:
from __future__ import annotations
import hashlib
import io
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

DATA_ROOT = Path('/home/slow_data/Air_Quality')
LANDSCAN_DIR = DATA_ROOT / 'LandScan'
FIRMS_ZIP    = DATA_ROOT / 'DL_FIRE_M-C61_759406.zip'

def md5_path(p: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with p.open('rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

## 1. LandScan ZIP-level fingerprint

If the MD5 column is identical across all three rows, the ZIPs are byte-equal — the downloader saved the same payload under different year-stamped names.

In [2]:
rows = []
for p in sorted(LANDSCAN_DIR.glob('landscan-mosaic-vietnam-*.zip')):
    rows.append({
        'file':  p.name,
        'bytes': p.stat().st_size,
        'md5':   md5_path(p),
    })
ls_zip = pd.DataFrame(rows)
display(ls_zip)

if ls_zip['md5'].nunique() == 1:
    print('⚠️  All LandScan ZIPs are byte-identical — every year filename points to the same payload.')
else:
    print('✓ LandScan ZIPs differ by content.')

,file,bytes,md5
0,landscan-mosaic-vietnam-v1-assets-2022.zip,120641124,afb1c546cd8d7e297083a0616eb8b6d5
1,landscan-mosaic-vietnam-v1-assets-2023.zip,120641124,afb1c546cd8d7e297083a0616eb8b6d5
2,landscan-mosaic-vietnam-v1-assets-2024.zip,120641124,afb1c546cd8d7e297083a0616eb8b6d5


⚠️  All LandScan ZIPs are byte-identical — every year filename points to the same payload.


## 2. LandScan raster payload check

Even when ZIPs differ, the GeoTIFF inside may carry the same year's data with a different acquisition timestamp. We:
1. Open each ZIP, find the .tif inside.
2. Read its raster array and hash it.
3. Report the bounds and a 16-bin histogram of population values so visual inspection can confirm year-to-year differences (population grows, rural-to-urban shifts happen).

In [3]:
import rasterio

raster_rows = []
for zip_path in sorted(LANDSCAN_DIR.glob('landscan-mosaic-vietnam-*.zip')):
    with zipfile.ZipFile(zip_path) as zf:
        tifs = [n for n in zf.namelist() if n.lower().endswith('.tif')]
        for tif_name in tifs:
            with zf.open(tif_name) as fh:
                buf = fh.read()
            tif_hash = hashlib.md5(buf).hexdigest()
            with rasterio.MemoryFile(buf) as mem, mem.open() as src:
                arr = src.read(1)
                bounds = tuple(round(v, 3) for v in src.bounds)
                nodata = src.nodata
                valid = arr if nodata is None else np.where(arr == nodata, np.nan, arr)
                valid = np.asarray(valid, dtype=np.float64)
                with np.errstate(invalid='ignore'):
                    pop_total = float(np.nansum(valid))
                    pop_max   = float(np.nanmax(valid))
                raster_rows.append({
                    'zip':      zip_path.name,
                    'tif':      tif_name,
                    'shape':    arr.shape,
                    'bounds':   bounds,
                    'pop_sum':  pop_total,
                    'pop_max':  pop_max,
                    'tif_md5':  tif_hash,
                })

ls_rast = pd.DataFrame(raster_rows)
display(ls_rast)

if ls_rast.empty:
    print('No .tif files found inside the LandScan zips — inspect the ZIP contents manually.')
elif ls_rast['tif_md5'].nunique() == 1:
    print('⚠️  All LandScan rasters have identical content — only one year of population data is actually present.')
    print('   → Action: re-download per-year by hitting the LandScan year-specific URLs.')
else:
    print(f"✓ Detected {ls_rast['tif_md5'].nunique()} distinct LandScan rasters across years.")

,zip,tif,shape,bounds,pop_sum,pop_max,tif_md5
0,landscan-mosaic-vietnam-v1-assets-2022.zip,landscan-mosaic-vietnam-v1.tif,"(17997, 8811)","(102.134, 8.405, 109.477, 23.402)",1.057590e+08,1814.311646,1572953946bb8e6e3b18b1896e04b521
1,landscan-mosaic-vietnam-v1-assets-2022.zip,landscan-mosaic-vietnam-v1-confidence-interval...,"(17997, 8811)","(102.134, 8.405, 109.477, 23.402)",4.041788e+06,1.000000,5ddb5cd3df9738ed9032a6731569eeb6
2,landscan-mosaic-vietnam-v1-assets-2022.zip,landscan-mosaic-vietnam-v1-colorized.tif,"(17997, 8811)","(102.134, 8.405, 109.477, 23.402)",1.757479e+09,255.000000,184a260d23cd287c95ce23e6e16ec83c
3,landscan-mosaic-vietnam-v1-assets-2023.zip,landscan-mosaic-vietnam-v1.tif,"(17997, 8811)","(102.134, 8.405, 109.477, 23.402)",1.057590e+08,1814.311646,1572953946bb8e6e3b18b1896e04b521
4,landscan-mosaic-vietnam-v1-assets-2023.zip,landscan-mosaic-vietnam-v1-confidence-interval...,"(17997, 8811)","(102.134, 8.405, 109.477, 23.402)",4.041788e+06,1.000000,5ddb5cd3df9738ed9032a6731569eeb6
5,landscan-mosaic-vietnam-v1-assets-2023.zip,landscan-mosaic-vietnam-v1-colorized.tif,"(17997, 8811)","(102.134, 8.405, 109.477, 23.402)",1.757479e+09,255.000000,184a260d23cd287c95ce23e6e16ec83c
6,landscan-mosaic-vietnam-v1-assets-2024.zip,landscan-mosaic-vietnam-v1.tif,"(17997, 8811)","(102.134, 8.405, 109.477, 23.402)",1.057590e+08,1814.311646,1572953946bb8e6e3b18b1896e04b521
7,landscan-mosaic-vietnam-v1-assets-2024.zip,landscan-mosaic-vietnam-v1-confidence-interval...,"(17997, 8811)","(102.134, 8.405, 109.477, 23.402)",4.041788e+06,1.000000,5ddb5cd3df9738ed9032a6731569eeb6
8,landscan-mosaic-vietnam-v1-assets-2024.zip,landscan-mosaic-vietnam-v1-colorized.tif,"(17997, 8811)","(102.134, 8.405, 109.477, 23.402)",1.757479e+09,255.000000,184a260d23cd287c95ce23e6e16ec83c


✓ Detected 3 distinct LandScan rasters across years.


## 3. FIRMS shapefile inventory

MODIS C6.1 fire archive (`fire_archive_*.shp`) covers validated retrievals; `fire_nrt_*.shp` covers near-real-time and is overwritten when records get promoted to the archive. If both are present and overlap in time, the same fire can appear twice. We check:
1. ACQ_DATE coverage per shapefile.
2. Geographic bounds (sanity check that this is the Vietnam-region pull).
3. Exact (lat, lon, ACQ_DATE, ACQ_TIME) duplicate rows across the two shapefiles.

In [4]:
import geopandas as gpd

with zipfile.ZipFile(FIRMS_ZIP) as zf:
    shp_members = sorted({n.rsplit('.', 1)[0] for n in zf.namelist() if n.lower().endswith('.shp')})
    print('Shapefiles in the FIRMS bundle:', shp_members)

def load_firms_layer(stem: str) -> gpd.GeoDataFrame:
    return gpd.read_file(f'zip://{FIRMS_ZIP}!{stem}.shp')

firms_layers = {stem: load_firms_layer(stem) for stem in shp_members}
summary = []
for stem, gdf in firms_layers.items():
    gdf = gdf.copy()
    if 'ACQ_DATE' in gdf.columns:
        gdf['ACQ_DATE'] = pd.to_datetime(gdf['ACQ_DATE'], errors='coerce')
    summary.append({
        'layer':     stem,
        'n_records': len(gdf),
        'date_min':  gdf['ACQ_DATE'].min() if 'ACQ_DATE' in gdf.columns else None,
        'date_max':  gdf['ACQ_DATE'].max() if 'ACQ_DATE' in gdf.columns else None,
        'bbox':      tuple(round(v, 3) for v in gdf.total_bounds),
        'crs':       str(gdf.crs),
    })
    firms_layers[stem] = gdf

display(pd.DataFrame(summary))

Shapefiles in the FIRMS bundle: ['fire_archive_M-C61_759406', 'fire_nrt_M-C61_759406']


,layer,n_records,date_min,date_max,bbox,crs
0,fire_archive_M-C61_759406,271413,2022-09-01,2026-02-28,"(102.0, 8.03, 109.999, 23.5)",EPSG:4326
1,fire_nrt_M-C61_759406,33736,2026-03-01,2026-04-30,"(102.0, 8.674, 109.985, 23.498)",EPSG:4326


### Overlap test

We keep only the rows whose ACQ_DATE falls within the *intersection* of the two layers' date ranges, then check for exact (lat, lon, ACQ_DATE, ACQ_TIME) collisions. A non-zero collision count means we have to dedupe before counting active fires per cell.

In [5]:
if len(firms_layers) >= 2:
    arch_key = next((k for k in firms_layers if 'archive' in k.lower()), None)
    nrt_key  = next((k for k in firms_layers if 'nrt'     in k.lower()), None)
    if arch_key and nrt_key:
        arch = firms_layers[arch_key]
        nrt  = firms_layers[nrt_key]
        overlap_lo = max(arch['ACQ_DATE'].min(), nrt['ACQ_DATE'].min())
        overlap_hi = min(arch['ACQ_DATE'].max(), nrt['ACQ_DATE'].max())
        print(f'archive coverage : {arch["ACQ_DATE"].min().date()} … {arch["ACQ_DATE"].max().date()}')
        print(f'nrt     coverage : {nrt["ACQ_DATE"].min().date()}  … {nrt["ACQ_DATE"].max().date()}')
        print(f'overlap window   : {overlap_lo.date() if pd.notna(overlap_lo) else None} … {overlap_hi.date() if pd.notna(overlap_hi) else None}')

        if pd.notna(overlap_lo) and overlap_lo <= overlap_hi:
            join_cols = [c for c in ('LATITUDE', 'LONGITUDE', 'ACQ_DATE', 'ACQ_TIME') if c in arch.columns and c in nrt.columns]
            a = arch[(arch['ACQ_DATE'] >= overlap_lo) & (arch['ACQ_DATE'] <= overlap_hi)][join_cols]
            n = nrt [(nrt ['ACQ_DATE'] >= overlap_lo) & (nrt ['ACQ_DATE'] <= overlap_hi)][join_cols]
            collisions = a.merge(n, on=join_cols, how='inner')
            print(f'rows in overlap window — archive : {len(a):,}, nrt : {len(n):,}')
            print(f'exact-key collisions             : {len(collisions):,}')
            if len(collisions) > 0:
                print('   → Action: dedupe by (LATITUDE, LONGITUDE, ACQ_DATE, ACQ_TIME) before any per-day per-cell count.')
        else:
            print('✓ No date overlap between archive and nrt — safe to concatenate.')

archive coverage : 2022-09-01 … 2026-02-28
nrt     coverage : 2026-03-01  … 2026-04-30
overlap window   : 2026-03-01 … 2026-02-28
✓ No date overlap between archive and nrt — safe to concatenate.


## 4. Suggested resolution

- **If LandScan rasters are identical**, only one year of population data is on disk. Stage B can either (a) treat population as a static layer (acceptable per §7.8 — GPWv4 was already going to be a 2020 snapshot), or (b) re-pull per year from the LandScan year-specific download URLs.
- **If FIRMS overlap is non-zero**, the Stage B FIRMS loader will dedupe on `(LATITUDE, LONGITUDE, ACQ_DATE, ACQ_TIME)` before counting active fires per cell per day. Recording the assumption here so the loader can reference it.
- This notebook does **not** modify any files on disk. It only reports — fix actions are taken by the loaders in `ancillary.py`.